Alerta de ordens pendentes criação de KPI:<br>
ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por email e telefone de quem tem o cadastro completo 

In [0]:
import pyspark.sql.functions as F

# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
df1 = spark.read.table("bikestore.logistics.silver_customers")
df2 = spark.read.table("bikestore.logistics.silver_orders")

In [0]:
df_orders_gold = df2.select('customer_id','order_date','quantity','store_name')\
    .filter(F.col('status') == "Pending")\
    .filter((F.col("shipped_date").isNotNull()))\
    .groupBy('customer_id','store_name','order_date')\
    .agg(F.sum('quantity').alias('quantity'))

In [0]:
df_customers = df1.select('first_name','email','phone','customer_id')

In [0]:
df_final =(df_orders_gold.join(df_customers, on="customer_id", how="left")
    .filter(F.col('email').isNotNull())
    .filter(F.col('phone').isNotNull())
)

In [0]:
#salvar arquivo parquet na silver como delta
df_final.write\
.format('delta')\
.mode('overwrite')\
.option("mergeSchema", "true")\
.save(gold_path+'orders_pending')


In [0]:
#criando tabela
df = df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.gold_orders_pending")

In [0]:
%sql
select * from bikestore.logistics.gold_orders_pending

customer_id,order_date,quantity,store_name,first_name,email,phone
138,2018-04-27,4,Santa Cruz Bikes,Jone,jone.bernard@hotmail.com,(657) 536-5165
80,2018-04-22,5,Baldwin Bikes,Sarai,sarai.mckee@msn.com,(716) 912-8110
67,2018-04-17,1,Santa Cruz Bikes,Tommie,tommie.melton@gmail.com,(916) 802-2952
110,2018-04-13,3,Santa Cruz Bikes,Ollie,ollie.zimmerman@yahoo.com,(657) 648-2863
170,2018-04-17,4,Santa Cruz Bikes,Regine,regine.gonzales@gmail.com,(805) 763-4045
39,2018-04-17,8,Baldwin Bikes,Janetta,janetta.aguirre@aol.com,(717) 670-2634
56,2018-04-28,2,Rowlett Bikes,Lolita,lolita.mosley@hotmail.com,(281) 363-3309
5,2018-04-17,4,Santa Cruz Bikes,Charolette,charolette.rice@msn.com,(916) 381-6003
43,2018-04-29,8,Rowlett Bikes,Mozelle,mozelle.carter@aol.com,(281) 489-9656
